In [6]:
# 1. Importação de Pacotes Essenciais
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB

# 2. Carregando a Base de Dados direto do Google Sheets
url_planilha = 'https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/export?format=csv'
df_gripe = pd.read_csv(url_planilha)

# 3. Limpeza: Renomeando colunas para o padrão de mercado (Snake Case)
nomes_colunas = [
    'data_hora', 'ficou_gripado', 'tomou_vacina', 'frequentou_aglomeracao',
    'viajou_longe', 'tem_alergia', 'horas_sono', 'atividade_fisica',
    'alimentacao_balanceada', 'lavou_maos', 'nivel_estresse'
]
df_gripe.columns = nomes_colunas

# 4. Seleção de Atributos: Removendo colunas irrelevantes para a indução do modelo
df_gripe = df_gripe.drop(columns=['data_hora'])

print("Visão geral dos dados baixados direto da nuvem:")
display(df_gripe.head(3))

Visão geral dos dados baixados direto da nuvem:


,ficou_gripado,tomou_vacina,frequentou_aglomeracao,viajou_longe,tem_alergia,horas_sono,atividade_fisica,alimentacao_balanceada,lavou_maos,nivel_estresse
0,Sim,Sim,Sim,Poucas vezes por ano,Médio,4 horas ou menos,Sim,Às vezes,3 a 5 vezes,5.0
1,Sim,Sim,Sim,Nuca,Não,entre 4 e 6 horas,Não,"Não, raramente",Mais de 10 vezes,3.0
2,Sim,Não,Sim,Poucas vezes por ano,Pouco,mais de 6 horas,Sim,Às vezes,6 a 10 vezes,3.0


In [7]:
# 1. Tratamento de Valores Ausentes: Preenchimento automático pela Moda
# Esta técnica da Aula 02 garante que não percamos instâncias importantes
for col in df_gripe.columns:
    valor_frequente = df_gripe[col].mode()[0]
    df_gripe[col].fillna(valor_frequente, inplace=True)

# 2. Padronização de Tipos: Garantir que o nível de estresse seja tratado como texto categórico
df_gripe['nivel_estresse'] = df_gripe['nivel_estresse'].astype(int).astype(str)

# 3. Separação de Atributos: X (Características) e y (O que queremos prever)
X_caracteristicas = df_gripe.drop(columns=['ficou_gripado'])
y_alvo = df_gripe['ficou_gripado']

print(f"Limpeza concluída! Total de instâncias processadas: {len(df_gripe)}")
print(f"Valores nulos restantes: {df_gripe.isna().sum().sum()}")

Limpeza concluída! Total de instâncias processadas: 186
Valores nulos restantes: 0


/tmp/ipykernel_40090/4010375012.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_gripe[col].fillna(valor_frequente, inplace=True)


In [8]:
# Inicializando o codificador com proteção para categorias desconhecidas
tradutor_dados = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# Treinando o 'tradutor' com os nossos dados de treino
X_numerico = tradutor_dados.fit_transform(X_caracteristicas)

print("Exemplo de conversão (Vetor de Atributos):")
print(f"Original: {X_caracteristicas.iloc[0].values}")
print(f"Numérico: {X_numerico[0]}")

Exemplo de conversão (Vetor de Atributos):
Original: ['Sim' 'Sim' 'Poucas vezes por ano' 'Médio' '4 horas ou menos' 'Sim'
 'Às vezes' '3 a 5 vezes' '5']
Numérico: [1. 1. 2. 1. 0. 1. 2. 1. 4.]


In [9]:
# Instanciando o modelo com o Estimador de Laplace (alpha=1.0)
modelo_gripe = CategoricalNB(alpha=1.0)

# O processo de indução (Treino)
modelo_gripe.fit(X_numerico, y_alvo)

print("Indução concluída!")
print(f"O modelo aprendeu a classificar entre: {modelo_gripe.classes_}")

Indução concluída!
O modelo aprendeu a classificar entre: ['Não' 'Sim']


In [10]:
# Criando um perfil de teste com categorias que realmente existem no seu forms
perfil_teste = pd.DataFrame([{
    'tomou_vacina': 'Sim',
    'frequentou_aglomeracao': 'Sim',
    'viajou_longe': 'Poucas vezes por ano',
    'tem_alergia': 'Não',
    'horas_sono': 'mais de 6 horas',
    'atividade_fisica': 'Sim',
    'alimentacao_balanceada': 'Às vezes',  # <-- Corrigido aqui
    'lavou_maos': 'Mais de 10 vezes',
    'nivel_estresse': '2'
}])

# Aplicando a transformação numérica ao dado novo
perfil_convertido = tradutor_dados.transform(perfil_teste)

# Realizando a predição e calculando as probabilidades
resultado = modelo_gripe.predict(perfil_convertido)[0]
probabilidades = modelo_gripe.predict_proba(perfil_convertido)[0]

print("=== RELATÓRIO DO CLASSIFICADOR ===")
print(f"Previsão: FICOU GRIPADO? -> {resultado.upper()}\n")

print("Probabilidades Calculadas:")
for classe, prob in zip(modelo_gripe.classes_, probabilidades):
    print(f" - {classe}: {prob * 100:.2f}%")

=== RELATÓRIO DO CLASSIFICADOR ===
Previsão: FICOU GRIPADO? -> NÃO

Probabilidades Calculadas:
 - Não: 54.71%
 - Sim: 45.29%
